[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/rubber_band/rubber_band.ipynb)

# Rubber Band Heat Engine

This example computes a rubber band lifting a weight by heat, the Gough–Joule effect. A rubber band's tension comes from its entropy: stretching the band lines up its polymer chains, so a stretched band has less entropy than a slack one at the same temperature, and it pulls back in proportion to its absolute temperature. When a lamp warms a band that holds up a weight, the band pulls harder, shortens and lifts the weight; as it cools, it lengthens and lets the weight down again, acting as a heat engine that turns some of the lamp's heat into gravitational potential energy. Stretched quickly, the band warms; let go, it cools.

The band is modelled on a triangulated strip in the plane `x+ y+` joined with one degenerate quantity direction `s0` for entropy, and each vertex state is a point `places + entropies * entropy` combining its spatial offset with its entropy. Each triangle's energy depends on its corners' points, and its derivative at a vertex is a single plane of potentials, `temperatures * s - forces`, combining minus the elastic force in the plane with the absolute temperature along the entropy direction. Stretched at fixed entropy, the rubber warms directly from its equation of state, so the temperature rise sits in the triangles that stretch without a separate coupling term between mechanics and heat. Inside the band, thermal conduction is driven by `-gradient ^ s` through a positive conductivity, moving heat from warmer vertices to cooler ones without creating or destroying energy.

In tensor-calculus thermoelasticity, for the band with which [Feynman's lectures](https://www.feynmanlectures.caltech.edu/I_44.html) open their chapter on the laws of thermodynamics, the Helmholtz free energy per area $\psi(\mathbf F, T) = \frac{T}{T_0}\,W(\mathbf F) + c\,(T - T\ln T)$ separates the first Piola–Kirchhoff stress $\mathbf P = \partial\psi/\partial\mathbf F = \frac{T}{T_0}\,\partial W/\partial\mathbf F$ from the entropy density $s = -\partial\psi/\partial T = c\ln T - W(\mathbf F)/T_0$, coupling mechanical equilibrium $\nabla\cdot\mathbf P = \mathbf 0$ to the heat equation $c\,\partial_t T = \mathbf P : \partial_t\mathbf F + \nabla\cdot(\kappa\nabla T)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, dim, stack
from numga.sparse import spdiag
from examples.util import shapes
from examples.util.animation import show_states
from examples.thermodynamics.rubber_band import render, scenarios

# The plane `x+ y+` joined with one degenerate quantity direction `s0` for entropy.
space = Algebra("x+y+")
quantities = Algebra("s0")
ga = space * quantities
mv = NumpyContext(ga, units=True).multivector

Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# Spatial planes reading a place across and up the band, their dual spatial offsets, and thermal drives.
Upright = Plane.derive.within(space)                                          # Vector+
Spatial = Upright.derive.dual                                                 # Antivector-
Drive = ga.gatype.bivector().derive.touching(space, quantities)               # Bivector+
# Unit spatial directions across and up the plane, and a unit of entropy.
across = mv.x.dual()                                                          # [] Spatial-
up = mv.y.dual()                                                              # [] Spatial-
directions = stack((across, up))                                              # [2] Spatial-
entropy = mv.s.dual()                                                         # [] Point-

## The band at rest

The band hangs from a horizontal support clamped along its top row, triangulated in staggered rows of nearly equilateral triangles so that the strip favours neither diagonal, with its bottom rows forming the hanging weight. Lengths are measured in band widths, temperatures in units of the room's absolute temperature, and forces in units of the room-temperature shear modulus times the width, with gravitational acceleration equal to one. Each triangle's area and corner simplex gradients are computed once on this unloaded shape.

In [ ]:
# Build the unloaded hanging strip, mark the bottom rows as the weight, and clamp the top row.
n_across, n_band_rows, n_weight_rows = scenarios.n_across, scenarios.n_band_rows, scenarios.n_weight_rows
coordinates = scenarios.lattice(n_across, n_band_rows + n_weight_rows)        # [V, 2]
topology, coordinates = shapes.delaunay(coordinates)
V, E, F = topology.n_elements
rest = (mv(Upright, coordinates) * dim.length).dual().field()                 # Spatial-[V], length
band_length = n_band_rows * np.sqrt(3) / 2 / (n_across - 1)
in_weight = coordinates[topology.elements[2]][..., 1].max(axis=-1) < -band_length + 1e-9   # [F]
held = np.flatnonzero(coordinates[:, 1] == 0)                                 # [V_held]
corners = topology.incidence[2, 0]                                            # [3] Scalar+[F <- V]
corner_mean = corners.mean(axis=0)                                            # Scalar+[F <- V]
# Joining two oriented sides with a unit of entropy gives twice each triangle's signed area.
sides = topology.boundary[2, 1] * (topology.d[0] * rest)                      # [3] Spatial-[F]
double_areas = (sides[0] & sides[1]) & entropy                                # Scalar-[F]
areas = double_areas.norm() / 2                                               # Scalar+[F]
# Each corner's simplex gradient joins the facing side with entropy and divides by twice the signed area.
gradients = (entropy & sides) / double_areas                                  # [3] Upright+[F]
render.draw_band(topology, in_weight, rest);

## Material stiffness

The rubber is modelled as an isotropic plane-elastic material: a work form on pairs of displacement gradients built from dilation and rotation-free shear tractions, with a Poisson ratio of one third so the strip narrows visibly when stretched. Evaluating that work form on each triangle's corner simplex gradients gives its room-temperature stiffness map from corner displacements to corner forces. The bottom weight triangles use the same law with higher density and stiffness, and scaling both Lamé moduli with temperature makes the restoring pull proportional to temperature at fixed stretch.

In [ ]:
# Room-temperature Lamé moduli and per-triangle stiffening and density.
lame_lambda, lame_mu = scenarios.lame_lambda, scenarios.lame_mu
weighting = mv.scalar(in_weight[:, None] * 1.0).field()                       # Scalar+[F]
stiffening = 1 + (scenarios.weight_stiffening - 1) * weighting                # Scalar+[F]
densities = scenarios.density + (scenarios.weight_density - scenarios.density) * weighting   # Scalar+[F]
# Isotropic plane-elastic tractions on displacement gradients and the resulting symmetric work form.
dilation_traction = Upright * (Upright & Spatial)                             # [] Upright+ <- (Upright+, Upright+, Spatial-)
direct_shear = (Upright | Upright) * Spatial.dual_inverse()                   # [] Upright+ <- (Upright+, Upright+, Spatial-)
shear_traction = 2 * direct_shear - (Upright | (Upright ^ Spatial.dual_inverse()))  # [] Upright+ <- (Upright+, Upright+, Spatial-)
material = (lame_lambda * dilation_traction + lame_mu * shear_traction) & Spatial   # [] Scalar+ <- (Upright+, Upright+, Spatial-, Spatial-)
# Evaluating the work form on each triangle's corner gradients gives its room-temperature stiffness map.
bound = gradients.batch()                                                     # [3, F] Upright+
work = material(bound[:, None], bound[None, :], Spatial, Spatial)             # [3, 3, F] Scalar+ <- (Spatial-, Spatial-)
elastic = ((Upright & Spatial).solve(work) * (stiffening * areas).batch()).field()   # [3, 3] (Upright+ <- Spatial-)[F]
# Each vertex cell receives one third of the area and mass of its incident triangles.
dual_areas = corner_mean.adjugate() * areas                                   # Scalar+[V]
masses = corner_mean.adjugate() * (densities * areas)                         # Scalar+[V]

## Heat conduction and exchange

Each vertex point carries its cell's entropy alongside its spatial position. While switched on, the lamp supplies heat that falls off with the square of the distance from the bulb, warming the near side of the band most strongly, while the surrounding room removes heat by Newton's law of cooling. Inside the band, each corner's simplex gradient wedged with `s` forms a unit thermal drive, which the rubber's isotropic conductivity maps to an entropy flux; pulling that response back through the corner gradients assembles a symmetric, positive conduction map on vertex temperatures whose rows sum to zero.

In [ ]:
# Thermal parameters, cooling rate, and lamp position and intensity.
heat_capacity, conductivity = scenarios.heat_capacity, scenarios.conductivity
room_temperature, cooling = scenarios.room_temperature, scenarios.cooling
lamp_heat, reach = scenarios.lamp_heat, scenarios.reach
lamp = scenarios.lamp_across * across + scenarios.lamp_up * up                # [] Point-
capacities = heat_capacity * dual_areas                                       # Scalar+[V]
# Isotropic unit thermal conductivity: each spatial direction joined with entropy pairs with a drive to carry entropy.
sweeps = directions & entropy                                                 # [2] Antibivector-
spreading = (sweeps * (sweeps & Drive)).sum(axis=0)                           # [] Antibivector- <- Bivector+
# Pulling the unit conductivity back through the corner gradients assembles the vertex conduction map.
drives = -gradients ^ mv.s                                                    # [3] Bivector+[F]
local = areas * (mv.s & (gradients[:, None] ^ spreading(drives[None, :])))    # [3, 3] Scalar+[F]
conduction = conductivity * (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[V <- V]

## State and potentials

On each triangle, the entropy equals the thermal entropy (proportional to the logarithm of the temperature relative to room temperature) minus the room-temperature elastic energy divided by room temperature. At fixed entropy, stretching a triangle therefore raises its temperature exponentially with the elastic energy and scales its restoring forces by the same temperature ratio. Gathering both contributions onto the vertices yields one plane of potentials per vertex, combining minus the elastic force with the temperature.

In [ ]:
def potentials(points: Point) -> Plane:
    """Potentials conjugate to the vertex state, combining minus the elastic force in the plane with the absolute
    temperature along the entropy direction."""
    # Separate each vertex's entropy from its spatial shift relative to rest.
    held_entropy = mv.s & points                                              # Scalar+[V]
    shifts = corners * (points.cast(Spatial) - rest)                          # [3] Spatial-[F]
    # Room-temperature elastic forces on the corners and the corresponding elastic energy on each triangle.
    springs = -elastic(shifts[None, :]).sum(axis=1)                           # [3] Upright+[F]
    stretch = -(springs & shifts).sum(axis=0) / 2                             # Scalar+[F]
    # Mean vertex entropy density and stretch determine each triangle's temperature ratio relative to the room.
    entropy_densities = corner_mean * (held_entropy / dual_areas)             # Scalar+[F]
    warmth = ((entropy_densities + stretch / (areas * room_temperature)) / heat_capacity).exp()   # Scalar+[F]
    # Area-average the triangle temperatures onto the vertex cells and scale the corner forces by `warmth`.
    temperatures = room_temperature * (corner_mean.adjugate() * (areas * warmth)) / dual_areas   # Scalar+[V]
    forces = (corners.adjugate() * (warmth * springs)).sum(axis=0)            # Upright+[V]
    return temperatures * mv.s - forces

## Stepping the band

Each time step splits into a reversible mechanical step and an irreversible thermal step. Position Verlet advances the free vertices at fixed entropy using the forces evaluated at the midpoint of the step, damped by air drag, and heat from the lamp, the room and conduction then shifts each vertex state along the entropy direction according to the logarithmic entropy gain of its cell.

In [ ]:
# Gravitational pull, downward hand pull on the weight, air drag, time step, and projection onto the free vertices.
pull = -scenarios.gravity * mv.y                                              # [] Upright+
hand = scenarios.hand_pull * (corner_mean.adjugate() * (weighting * densities * areas)) * pull   # Upright+[V]
drag, dt = scenarios.drag, scenarios.dt
free = topology.reading(np.setdiff1d(np.arange(V), held), 0)                  # Scalar+[V_free <- V]
freeing = free.adjugate() * free                                              # Scalar+[V <- V]


def move(points: Point, velocities: Spatial, forces: Upright, pulled: float) -> tuple[Point, Spatial]:
    """Vertex state and velocities after a mechanical step at fixed entropy under midpoint elastic forces, gravity,
    air drag and the downward hand pull, with the clamped top row held still."""
    # Implicit midpoint air drag updates the free velocities before advancing the positions by the mean velocity.
    loads = forces + masses * pull + pulled * hand                            # Upright+[V]
    kicked = freeing * (((1 - drag * dt / 2) * velocities + dt * (loads / masses).dual()) / (1 + drag * dt / 2))   # Spatial-[V]
    return points + dt * (velocities + kicked) / 2, kicked

In [ ]:
def exchange(points: Point, temperatures: Scalar, lit: float) -> Scalar:
    """Entropy gained by each vertex cell over one step from lamp heating, Newton cooling to the room, and heat
    conduction across the band."""
    # Lamp intensity falls off with squared spatial distance after projecting out the entropy component from `points`.
    offsets = points.cast(Spatial) - lamp                                     # Spatial-[V]
    exposures = 1 / (1 + (offsets.dual_inverse() & offsets) / reach ** 2)     # Scalar+[V]
    exchanged = dt * dual_areas * (lit * lamp_heat * exposures - cooling * (temperatures - room_temperature))   # Scalar+[V]
    conducted = -dt * (conduction * temperatures)                             # Scalar+[V]
    # Adding heat to a cell of constant heat capacity raises its entropy logarithmically with the temperature ratio.
    return capacities * (1 + (exchanged + conducted) / (capacities * temperatures)).log()

In [ ]:
def step(points: Point, velocities: Spatial, lit: float, pulled: float) -> tuple[Point, Spatial]:
    """Vertex state and velocities after one step, combining a reversible mechanical step at fixed entropy with an
    irreversible entropy shift from the lamp, the room and heat conduction."""
    # Evaluate the potentials at the midpoint to split out the temperatures and the elastic forces.
    middle = points + dt / 2 * velocities                                     # Point-[V], length | entropy
    middle_potentials = potentials(middle)                                    # Plane+[V], force | temperature
    temperatures = middle_potentials & entropy                                # Scalar+[V], temperature
    forces = (-middle_potentials).cast(Upright)                               # Upright+[V], extent*length*mass/time^2
    points, velocities = move(points, velocities, forces, pulled)
    gained = exchange(middle, temperatures, lit)                              # Scalar+[V], extent*length^2*mass/(temperature*time^2)
    return points + gained * entropy, velocities


def run(points: Point, exposure: np.ndarray, pulling: np.ndarray, n_stride: int
        ) -> Iterator[tuple[Point, Scalar, float]]:
    """Yield the band's vertex state, vertex temperatures and lamp state at the start and after each frame stride."""
    velocities = mv.zeros(Spatial, points.sites.count).field()                # Spatial-[V], 0
    yield points, potentials(points) & entropy, exposure[0]
    for exposures, pulls in zip(exposure.reshape(-1, n_stride), pulling.reshape(-1, n_stride)):
        for lit, pulled in zip(exposures, pulls):
            points, velocities = step(points, velocities, lit, pulled)
        yield points, potentials(points) & entropy, exposures[-1]

## Lifting and lowering the weight

Released from rest under the weight, the band first settles with the lamp off; the lamp then switches on for two and a half thermal relaxation times and switches off for as long. Because a real rubber band warms over minutes while its weight bounces within a second, the simulation compresses the warming time to several bounce periods, damped by air drag, and heats the rubber strongly enough to make the lift plain. This cell takes about fifteen seconds, most of it stepping the band.

In [ ]:
# Settling steps, lit and dark step counts, and frame stride.
n_settling, n_lit, n_dark, n_stride = scenarios.n_settling, scenarios.n_lit, scenarios.n_dark, scenarios.n_stride

# Settle the band under the weight at room temperature before running the lamp cycle.
*_, (settled, *_) = run(rest, np.zeros(n_settling), np.zeros(n_settling), n_settling)
exposure = np.concatenate([np.ones(n_lit), np.zeros(n_dark)])
states = show_states(run(settled, exposure, np.zeros_like(exposure), n_stride), render.animate, topology,
                     in_weight, lamp, room_temperature, scenarios.lamp_temperature, name="rubber_band",
                     duration_ms=scenarios.frame_duration_ms)

In the animation, the band starts grey at room temperature, narrowed below its support under the weight, and turns orange and red from the side nearest the glowing lamp. As it warms, it stiffens and lifts the weight; because the side facing the lamp is warmer and stiffer than the far side, the strip curves slightly toward the lamp and tilts the weight. After the lamp switches off, the band cools and lowers the weight again.

## Pulling the band

Driven mechanically, the same material law warms the band when stretched. With the lamp off, a downward hand pull equal to the hanging weight rises quickly, stretching the band at nearly constant entropy so its triangles warm directly from their equation of state, most strongly near the clamp where the strain is largest. While held down, the band loses that extra heat to the room; when released, it contracts quickly and cools below room temperature before warming back. This cell takes about ten seconds, most of it stepping the band.

In [ ]:
# Raise the downward hand pull smoothly, hold while heat leaks to the room, lower it smoothly, and leave to equilibrate.
rise = (1 - np.cos(np.pi * np.arange(scenarios.n_rise) / scenarios.n_rise)) / 2
pulling = np.concatenate([rise, np.ones(scenarios.n_hold), rise[::-1], np.zeros(scenarios.n_hold)])
pulled_states = show_states(run(settled, np.zeros_like(pulling), pulling, n_stride), render.animate_pulled,
                            topology, in_weight, room_temperature, scenarios.pull_swing, scenarios.pull_drop,
                            name="rubber_band_pulled", duration_ms=scenarios.frame_duration_ms)

The animation uses the same palette from blue below room temperature through grey to red above it, spanning roughly one thousandth of the lamp's temperature range because a mechanical pull warms the band by a small fraction of a degree. The strip turns red during the pull, fades toward grey while held, turns blue on release, and returns toward grey as the room warms it back.

In [ ]:
# checks
# Doubling every cell's absolute temperature at fixed stretch doubles the band's pull on the support.
last_points, *_ = states[-1]
support = topology.reading(held, 0)                                           # Scalar+[V_held <- V]
warmer_points = last_points + heat_capacity * dual_areas * np.log(2) * entropy   # Point-[V]
held_potentials = [support * potentials(points) for points in (last_points, warmer_points)]   # Plane+[V_held]
support_pulls = [(-plane).cast(Upright).sites.sum() & up for plane in held_potentials]
assert (support_pulls[1] / support_pulls[0] - 2).abs() < 1e-12
# The weight rises on every frame while the lamp is lit, and descends more than three quarters of the way back.
weight_vertices = topology.reading(np.unique(topology.elements[2][in_weight]), 0)   # Scalar+[V_weight <- V]
n_frames = len(states)
heights = stack([(mv.y & (weight_vertices * points)).sites.mean() for points, *_ in states])   # [n_frames] Scalar+
lamp_on = np.array([on for *_, on in states]) > 0                             # [n_frames]
n_lit_frames = lamp_on.sum()
lifted = heights[lamp_on]                                                     # [n_lit_frames] Scalar+
assert (lifted[1:] > lifted[:-1]).all()
assert lifted[-1] - heights[-1] > 3 / 4 * (lifted[-1] - heights[0])
# Pulling warms the band above room temperature, and releasing cools it below room temperature.
pulled_changes = stack([(temperatures - room_temperature).sites.mean() for _, temperatures, _ in pulled_states])
let_go = len(pulled_states) // 2
assert pulled_changes[:let_go].max() > 0 > pulled_changes[let_go:].min()
# Both runs relax back within one fifth of their peak temperature excursion from the room.
lamp_changes = stack([(temperatures - room_temperature).sites.mean() for _, temperatures, _ in states])
for changes in (lamp_changes, pulled_changes):
    assert changes[-1].abs() < changes.abs().max() / 5